# `running_an_example`

This notebook simulates a 1,000-female cohort for 3 years and one healthy 30-year-old female for 3 years using the existing `hormone_cycler` simulator.

Design choices used here:

- Ages are deliberately balanced across 12-52 years so the cohort spans the full requested range.
- Medication probabilities are taken from U.S. National Survey of Family Growth estimates when the simulator can represent the method directly.
- PCOS and dysmenorrhea prevalences are mapped onto the simulator's supported factor flags using published prevalence data.
- Explicit `peri_menarche` and `perimenopause` subgroup flags are left off because the simulator already embeds age effects in its baseline cycle model, and enabling those flags across the general population would double count stage effects.
- Participant irregularity uses the Apple Women's Health Study participant-level estimand: mean adjacent-cycle difference of at least seven days. The table footnote omits “absolute,” but the methods define adjacent-cycle differences as absolute values; the notebook follows that convention.


In [1]:
from __future__ import annotations

import csv
import math
import os
import random
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
MPLCONFIGDIR = Path("/tmp") / "catamenial-epilepsy-sim-mplconfig"
CACHE_DIR = Path("/tmp") / "catamenial-epilepsy-sim-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

import matplotlib.pyplot as plt
import pandas as pd

OUTPUT_DIR = ROOT / "examples" / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXAMPLE1000_PATH = ROOT / "example1000.csv"
ONE_HEALTHY_PATH = ROOT / "oneHealthy.csv"
COHORT_OVERVIEW_FIG = OUTPUT_DIR / "running_example_cohort_overview.png"
COHORT_AGE_FIG = OUTPUT_DIR / "running_example_age_patterns.png"
HEALTHY_FIG = OUTPUT_DIR / "running_example_one_healthy.png"
HEALTHY_KINETICS_FIG = OUTPUT_DIR / "running_example_hormone_kinetics.png"

SEED = 20260315
NUM_PATIENTS = 1000
DAYS = 365 * 3

LARC_TO_IUD_SHARE = 8.1 / 10.5
LNG_IUD_SHARE = 541_635 / (541_635 + 99_389)
COPPER_IUD_SHARE = 1.0 - LNG_IUD_SHARE
PCOS_PREVALENCE = 0.066
DYSMENORRHEA_PREVALENCE = 0.20

US_INPUT_CITATIONS = {
    "daniels_abma_2025": {
        "label": "Daniels and Abma 2025",
        "reference": (
            "Daniels K, Abma JC. Current contraceptive status among females ages 15-49: "
            "United States, 2022-2023. NCHS Data Brief. 2025 Aug;(539):1-11. "
            "doi:10.15620/cdc/174618."
        ),
        "url": "https://www.cdc.gov/nchs/products/databriefs/db539.htm",
    },
    "azziz_2004": {
        "label": "Azziz et al. 2004",
        "reference": (
            "Azziz R, Woods KS, Reyna R, Key TJ, Knochenhauer ES, Yildiz BO. "
            "The prevalence and features of the polycystic ovary syndrome in an unselected "
            "population. Journal of Clinical Endocrinology & Metabolism. 2004;89(6):2745-2749. "
            "doi:10.1210/jc.2003-032046."
        ),
        "url": "https://pubmed.ncbi.nlm.nih.gov/15181052/",
    },
    "ju_2014": {
        "label": "Ju et al. 2014",
        "reference": (
            "Ju H, Jones M, Mishra G. The prevalence and risk factors of dysmenorrhea. "
            "Epidemiologic Reviews. 2014;36:104-113. doi:10.1093/epirev/mxt009."
        ),
        "url": "https://pubmed.ncbi.nlm.nih.gov/24284871/",
    },
    "marcus_2020": {
        "label": "Marcus et al. 2020",
        "reference": (
            "Marcus JL, Snowden JM, Murray Horwitz ME, et al. Use of intrauterine devices and "
            "risk of human immunodeficiency virus acquisition among insured women in the United "
            "States. Clinical Infectious Diseases. 2020;70(10):2221-2223. doi:10.1093/cid/ciz791."
        ),
        "url": "https://pubmed.ncbi.nlm.nih.gov/31412356/",
    },
}

from hormone_cycler.literature import CITATIONS
from hormone_cycler.model import simulate_diary
from hormone_cycler.types import MedicalFactors


In [2]:
def balanced_ages(num_patients: int, seed: int) -> list[float]:
    """Return a balanced age spread across the requested 12-52 year range."""

    rng = random.Random(seed)
    ages = list(range(12, 53))
    base = num_patients // len(ages)
    remainder = num_patients % len(ages)
    values: list[float] = []
    for index, age in enumerate(ages):
        count = base + (1 if index < remainder else 0)
        for _ in range(count):
            jitter = rng.uniform(-0.45, 0.45)
            values.append(round(min(52.0, max(12.0, age + jitter)), 1))
    rng.shuffle(values)
    return values


def contraceptive_probabilities(age_years: float) -> dict[str, float]:
    """Return age-specific medication probabilities from NSFG 2022-2023."""

    if age_years < 15.0 or age_years >= 50.0:
        return {"pill": 0.0, "hormonal_iud": 0.0, "copper_iud": 0.0}

    if age_years < 20.0:
        pill = 0.142
        larc = 0.046
    elif age_years < 30.0:
        pill = 0.168
        larc = 0.138
    elif age_years < 40.0:
        pill = 0.090
        larc = 0.124
    else:
        pill = 0.069
        larc = 0.081

    iud_total = larc * LARC_TO_IUD_SHARE
    return {
        "pill": pill,
        "hormonal_iud": iud_total * LNG_IUD_SHARE,
        "copper_iud": iud_total * COPPER_IUD_SHARE,
    }


def assign_medical_factors(age_years: float, rng: random.Random) -> MedicalFactors:
    """Sample one factor profile using published prevalence inputs."""

    probabilities = contraceptive_probabilities(age_years)
    draw = rng.random()
    oral_contraceptive_mode = None
    hormonal_iud = False
    copper_iud = False

    if draw < probabilities["pill"]:
        oral_contraceptive_mode = "cyclic"
    elif draw < probabilities["pill"] + probabilities["hormonal_iud"]:
        hormonal_iud = True
    elif draw < probabilities["pill"] + probabilities["hormonal_iud"] + probabilities["copper_iud"]:
        copper_iud = True

    return MedicalFactors(
        pcos=rng.random() < PCOS_PREVALENCE,
        oral_contraceptive_mode=oral_contraceptive_mode,
        hormonal_iud=hormonal_iud,
        copper_iud=copper_iud,
        dysmenorrhea=rng.random() < DYSMENORRHEA_PREVALENCE,
    )


def flatten_daily_row(row: dict[str, object]) -> dict[str, object]:
    """Expand nested medical factor dictionaries into analysis-friendly columns."""

    factors = row.pop("medical_factors", {})
    flat = dict(row)
    flat["pcos"] = int(bool(factors.get("pcos", False)))
    flat["oral_contraceptive_mode"] = factors.get("oral_contraceptive_mode") or "none"
    flat["hormonal_iud"] = int(bool(factors.get("hormonal_iud", False)))
    flat["copper_iud"] = int(bool(factors.get("copper_iud", False)))
    flat["perimenopause"] = int(bool(factors.get("perimenopause", False)))
    flat["peri_menarche"] = int(bool(factors.get("peri_menarche", False)))
    flat["dysmenorrhea"] = int(bool(factors.get("dysmenorrhea", False)))
    if flat["oral_contraceptive_mode"] != "none":
        flat["medication_category"] = "oral_contraceptive"
    elif flat["hormonal_iud"]:
        flat["medication_category"] = "hormonal_iud"
    elif flat["copper_iud"]:
        flat["medication_category"] = "copper_iud"
    else:
        flat["medication_category"] = "none"
    return flat


def flatten_cycle_row(row: dict[str, object]) -> dict[str, object]:
    factors = row.pop("medical_factors", {})
    flat = dict(row)
    flat["pcos"] = int(bool(factors.get("pcos", False)))
    flat["oral_contraceptive_mode"] = factors.get("oral_contraceptive_mode") or "none"
    flat["hormonal_iud"] = int(bool(factors.get("hormonal_iud", False)))
    flat["copper_iud"] = int(bool(factors.get("copper_iud", False)))
    flat["dysmenorrhea"] = int(bool(factors.get("dysmenorrhea", False)))
    return flat


def age_group(age_years: float) -> str:
    if age_years < 20.0:
        return "12-19"
    if age_years < 30.0:
        return "20-29"
    if age_years < 40.0:
        return "30-39"
    return "40-52"


def mean_absolute_adjacent_difference(cycle_lengths: list[int]) -> float:
    if len(cycle_lengths) < 2:
        return float("nan")
    diffs = [abs(right - left) for left, right in zip(cycle_lengths[:-1], cycle_lengths[1:])]
    return sum(diffs) / len(diffs)


## Published input assumptions

            | Input | Value used | Source / note |
| --- | --- | --- |
| Age range | Balanced spread from 12.0 to 52.0 years | User requested ages spanning 12-52; balanced rather than U.S.-weighted so all age ranges are represented. |
| Oral contraceptive prevalence | 15-19: 14.2%; 20-29: 16.8%; 30-39: 9.0%; 40-49: 6.9%; outside 15-49: 0% | NSFG 2022-2023 pill use among all U.S. females ages 15-49 (Daniels and Abma 2025). |
| IUD prevalence | Age-band LARC rate multiplied by IUD/LARC share (0.771); outside 15-49: 0% | Inference from Daniels and Abma 2025 because NSFG reports age-specific LARC use and overall IUD use separately. |
| Hormonal vs copper IUD split | 84.5% hormonal IUD, 15.5% copper IUD | Inference from U.S. insured IUD insertions in Marcus et al. 2020. |
| PCOS prevalence | 6.6% of patients | Azziz et al. 2004 U.S. unselected premenopausal sample. |
| Dysmenorrhea prevalence | 20% of patients | Conservative proxy for clinically meaningful dysmenorrhea, chosen inside the 2%-29% severe-pain range summarized by Ju et al. 2014. |
| Peri-menarche / perimenopause flags | Not explicitly sampled | The simulator already encodes age effects from Li et al. 2023 and related calibration. Leaving these subgroup flags off avoids double counting explicit stage modifiers. |

            Notes:

            - The age-specific IUD probabilities are an explicit inference: Daniels and Abma 2025 give age-specific LARC rates and overall IUD prevalence, so the notebook multiplies each age-band LARC rate by the overall IUD/LARC ratio.
            - The hormonal-versus-copper IUD split is also an inference from U.S. insertion data, because the NSFG brief does not break current IUD use out by device type.
            - The dysmenorrhea input is intentionally conservative because the simulator's `dysmenorrhea` flag represents a clinically relevant phenotype rather than any mild menstrual discomfort.


In [3]:
rng = random.Random(SEED)
ages = balanced_ages(NUM_PATIENTS, SEED)
profiles: list[dict[str, object]] = []
cycle_rows: list[dict[str, object]] = []

daily_fieldnames = [
    "patient_id",
    "day_index",
    "age_years",
    "cycle_index",
    "cycle_day",
    "cycle_length",
    "estradiol_pg_ml",
    "progesterone_ng_ml",
    "ovulation",
    "uterine_bleeding",
    "pcos",
    "oral_contraceptive_mode",
    "hormonal_iud",
    "copper_iud",
    "perimenopause",
    "peri_menarche",
    "dysmenorrhea",
    "medication_category",
]

with EXAMPLE1000_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=daily_fieldnames)
    writer.writeheader()
    for patient_index, age_years in enumerate(ages, start=1):
        factors = assign_medical_factors(age_years, rng)
        patient_seed = rng.randint(0, 2**31 - 1)
        patient_id = f"example-{patient_index:04d}"
        result = simulate_diary(
            days=DAYS,
            age_years=age_years,
            medical_factors=factors,
            seed=patient_seed,
            patient_id=patient_id,
        )
        profile = result.profile.to_dict()
        profile["age_group"] = age_group(float(profile["age_years"]))
        profile["pcos"] = int(bool(profile["medical_factors"]["pcos"]))
        profile["dysmenorrhea"] = int(bool(profile["medical_factors"]["dysmenorrhea"]))
        profile["oral_contraceptive_mode"] = profile["medical_factors"]["oral_contraceptive_mode"] or "none"
        profile["hormonal_iud"] = int(bool(profile["medical_factors"]["hormonal_iud"]))
        profile["copper_iud"] = int(bool(profile["medical_factors"]["copper_iud"]))
        if profile["oral_contraceptive_mode"] != "none":
            profile["medication_category"] = "oral_contraceptive"
        elif profile["hormonal_iud"]:
            profile["medication_category"] = "hormonal_iud"
        elif profile["copper_iud"]:
            profile["medication_category"] = "copper_iud"
        else:
            profile["medication_category"] = "none"
        profiles.append(profile)

        for cycle in result.cycles:
            cycle_row = flatten_cycle_row(cycle.to_dict())
            cycle_row["age_group"] = age_group(float(cycle_row["age_years"]))
            cycle_rows.append(cycle_row)

        for row in result.diary:
            writer.writerow(flatten_daily_row(row.to_dict()))

profiles_df = pd.DataFrame(profiles)
cycles_df = pd.DataFrame(cycle_rows)

patient_irregularity = (
    cycles_df.sort_values(["patient_id", "cycle_index"])
    .groupby("patient_id")["cycle_length"]
    .apply(lambda s: mean_absolute_adjacent_difference([int(value) for value in s.tolist()]))
    .reset_index(name="mean_absolute_adjacent_difference_days")
)
profiles_df = profiles_df.merge(patient_irregularity, on="patient_id", how="left")
profiles_df["irregular_participant"] = (
    profiles_df["mean_absolute_adjacent_difference_days"] >= 7.0
)

factor_summary_df = pd.DataFrame(
    [
        {"feature": "PCOS", "patients": int(profiles_df["pcos"].sum()), "percent": 100.0 * profiles_df["pcos"].mean()},
        {
            "feature": "Dysmenorrhea",
            "patients": int(profiles_df["dysmenorrhea"].sum()),
            "percent": 100.0 * profiles_df["dysmenorrhea"].mean(),
        },
        {
            "feature": "Cyclic oral contraceptive",
            "patients": int((profiles_df["oral_contraceptive_mode"] == "cyclic").sum()),
            "percent": 100.0 * (profiles_df["oral_contraceptive_mode"] == "cyclic").mean(),
        },
        {
            "feature": "Hormonal IUD",
            "patients": int(profiles_df["hormonal_iud"].sum()),
            "percent": 100.0 * profiles_df["hormonal_iud"].mean(),
        },
        {
            "feature": "Copper IUD",
            "patients": int(profiles_df["copper_iud"].sum()),
            "percent": 100.0 * profiles_df["copper_iud"].mean(),
        },
    ]
)

cohort_summary_df = pd.DataFrame(
    [
        {"metric": "Patients", "value": len(profiles_df)},
        {"metric": "Diary days per patient", "value": DAYS},
        {"metric": "Mean age (years)", "value": profiles_df["age_years"].mean()},
        {"metric": "Median age (years)", "value": profiles_df["age_years"].median()},
        {"metric": "Mean personal cycle length target (days)", "value": profiles_df["personal_cycle_mean_days"].mean()},
        {"metric": "Observed mean cycle length (days)", "value": cycles_df["cycle_length"].mean()},
        {"metric": "Observed ovulatory cycle rate", "value": cycles_df["ovulatory"].mean()},
        {"metric": "Observed mean bleeding days per cycle", "value": cycles_df["bleeding_days"].mean()},
        {"metric": "Mean absolute adjacent-cycle difference (days)", "value": profiles_df["mean_absolute_adjacent_difference_days"].dropna().mean()},
        {"metric": "Participants meeting the >=7-day irregularity definition", "value": profiles_df["irregular_participant"].mean()},
    ]
)

age_band_summary_df = (
    cycles_df.groupby("age_group")
    .agg(
        patients=("patient_id", "nunique"),
        cycles=("cycle_index", "count"),
        mean_cycle_length=("cycle_length", "mean"),
        ovulatory_cycle_rate=("ovulatory", "mean"),
        mean_bleeding_days=("bleeding_days", "mean"),
    )
    .reset_index()
)

medication_by_age_df = (
    profiles_df.groupby(["age_group", "medication_category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["none", "oral_contraceptive", "hormonal_iud", "copper_iud"], fill_value=0)
)

print(f"Wrote cohort diary CSV to {EXAMPLE1000_PATH}")
print(f"Simulated {len(profiles_df)} patients and {len(cycles_df)} complete or truncated cycles")


Wrote cohort diary CSV to /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/example1000.csv
Simulated 1000 patients and 38523 complete or truncated cycles


## Cohort summary statistics

            ### Overall cohort summary

            | metric | value |
| --- | --- |
| Patients | 1000 |
| Diary days per patient | 1095 |
| Mean age (years) | 31.812 |
| Median age (years) | 31.7 |
| Mean personal cycle length target (days) | 28.921 |
| Observed mean cycle length (days) | 29.165 |
| Observed ovulatory cycle rate | 0.787 |
| Observed mean bleeding days per cycle | 4.005 |
| Mean absolute adjacent-cycle difference (days) | 4.737 |
| Participants meeting the >=7-day irregularity definition | 0.185 |

            ### Assigned medications and medical problems

            | feature | patients | percent |
| --- | --- | --- |
| PCOS | 69 | 6.9 |
| Dysmenorrhea | 187 | 18.7 |
| Cyclic oral contraceptive | 88 | 8.8 |
| Hormonal IUD | 51 | 5.1 |
| Copper IUD | 5 | 0.5 |

            ### Cycle outcomes by age band

            | age_group | patients | cycles | mean_cycle_length | ovulatory_cycle_rate | mean_bleeding_days |
| --- | --- | --- | --- | --- | --- |
| 12-19 | 211 | 7995 | 29.688 | 0.68 | 4.065 |
| 20-29 | 246 | 9234 | 29.94 | 0.795 | 3.941 |
| 30-39 | 242 | 9408 | 28.89 | 0.857 | 4 |
| 40-52 | 301 | 11886 | 28.428 | 0.797 | 4.018 |


In [4]:
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(profiles_df["age_years"], bins=20, color="#4C72B0", edgecolor="white")
axes[0, 0].set_title("Age distribution")
axes[0, 0].set_xlabel("Age (years)")
axes[0, 0].set_ylabel("Patients")

axes[0, 1].bar(
    factor_summary_df["feature"],
    factor_summary_df["percent"],
    color=["#C44E52", "#55A868", "#8172B3", "#CCB974", "#64B5CD"],
)
axes[0, 1].set_title("Assigned condition and medication prevalence")
axes[0, 1].set_ylabel("Percent of patients")
axes[0, 1].tick_params(axis="x", rotation=25)

box_groups = [cycles_df.loc[cycles_df["age_group"] == label, "cycle_length"] for label in ["12-19", "20-29", "30-39", "40-52"]]
axes[1, 0].boxplot(
    box_groups,
    tick_labels=["12-19", "20-29", "30-39", "40-52"],
    patch_artist=True,
)
axes[1, 0].set_title("Cycle length by age band")
axes[1, 0].set_ylabel("Cycle length (days)")

ovulation_rates = age_band_summary_df["ovulatory_cycle_rate"] * 100.0
axes[1, 1].plot(age_band_summary_df["age_group"], ovulation_rates, marker="o", linewidth=2, color="#4C72B0")
axes[1, 1].set_title("Ovulatory cycle rate by age band")
axes[1, 1].set_ylabel("Ovulatory cycles (%)")
axes[1, 1].set_ylim(0, 100)

fig.suptitle("Cohort overview: 1,000 simulated females over 3 years", fontsize=15)
fig.tight_layout()
fig.savefig(COHORT_OVERVIEW_FIG, dpi=180, bbox_inches="tight")
plt.close(fig)

irregularity_by_age_df = (
    profiles_df.groupby("age_group")
    .agg(
        irregular_participant_prevalence=("irregular_participant", "mean"),
        mean_personal_target=("personal_cycle_mean_days", "mean"),
    )
    .reset_index()
)

medication_share_df = medication_by_age_df.div(medication_by_age_df.sum(axis=1), axis=0) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(
    irregularity_by_age_df["age_group"],
    irregularity_by_age_df["mean_personal_target"],
    marker="o",
    linewidth=2,
    color="#DD8452",
)
axes[0].set_title("Mean personal cycle target by age band")
axes[0].set_ylabel("Days")

left = pd.Series(0.0, index=medication_share_df.index)
colors = {
    "none": "#9A9A9A",
    "oral_contraceptive": "#8172B3",
    "hormonal_iud": "#64B5CD",
    "copper_iud": "#CCB974",
}
for column in medication_share_df.columns:
    axes[1].bar(
        medication_share_df.index,
        medication_share_df[column],
        bottom=left.values,
        label=column.replace("_", " "),
        color=colors[column],
    )
    left += medication_share_df[column]
axes[1].set_title("Medication mix by age band")
axes[1].set_ylabel("Percent of patients")
axes[1].legend(frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig(COHORT_AGE_FIG, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Wrote cohort figures to {COHORT_OVERVIEW_FIG} and {COHORT_AGE_FIG}")


Wrote cohort figures to /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/examples/reports/running_example_cohort_overview.png and /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/examples/reports/running_example_age_patterns.png


## Cohort figures

![Cohort overview](examples/reports/running_example_cohort_overview.png)

![Age-pattern figure](examples/reports/running_example_age_patterns.png)


In [5]:
healthy_result = simulate_diary(
    days=DAYS,
    age_years=30.0,
    medical_factors=MedicalFactors(),
    seed=SEED + 5000,
    patient_id="healthy-30y",
)

healthy_rows = [flatten_daily_row(row.to_dict()) for row in healthy_result.diary]
healthy_df = pd.DataFrame(healthy_rows)
healthy_df.to_csv(ONE_HEALTHY_PATH, index=False)

healthy_cycles_df = pd.DataFrame([flatten_cycle_row(cycle.to_dict()) for cycle in healthy_result.cycles])

complete_cycle_indices = [
    int(cycle_index)
    for cycle_index, group in healthy_df.groupby("cycle_index")
    if (
        len(group) == int(group["cycle_length"].iloc[0])
        and int(group["cycle_day"].iloc[0]) == 1
        and int(group["cycle_day"].iloc[-1]) == int(group["cycle_length"].iloc[0])
    )
]
complete_ovulatory_indices = [
    cycle_index
    for cycle_index in complete_cycle_indices
    if bool(healthy_cycles_df.loc[healthy_cycles_df["cycle_index"] == cycle_index, "ovulatory"].iloc[0])
]
kinetic_cycle_index = complete_ovulatory_indices[0]
kinetic_cycle_df = healthy_df.loc[healthy_df["cycle_index"] == kinetic_cycle_index].copy()
kinetic_following_df = healthy_df.loc[healthy_df["cycle_index"] == kinetic_cycle_index + 1].head(1)
kinetic_plot_df = pd.concat([kinetic_cycle_df, kinetic_following_df], ignore_index=True)

ovulation_day = int(kinetic_cycle_df.loc[kinetic_cycle_df["ovulation"] == 1, "cycle_day"].iloc[0])
follicular_e2 = kinetic_cycle_df.loc[kinetic_cycle_df["cycle_day"] <= ovulation_day, "estradiol_pg_ml"]
peak_width_days = int((follicular_e2 >= 0.80 * follicular_e2.max()).sum())
progesterone_values = kinetic_cycle_df["progesterone_ng_ml"].tolist()
progesterone_peak = max(progesterone_values)
progesterone_peak_day = progesterone_values.index(progesterone_peak) + 1
progesterone_plateau_days = sum(value >= 0.75 * progesterone_peak for value in progesterone_values)
progesterone_rise_day = next(
    day
    for day, value in enumerate(progesterone_values, start=1)
    if day >= ovulation_day and value >= 5.0
)
luteal_e2 = kinetic_cycle_df.loc[
    kinetic_cycle_df["cycle_day"] > ovulation_day, "estradiol_pg_ml"
]
luteal_e2_peak_ratio = float(luteal_e2.max() / follicular_e2.max())
withdrawal_transitions = 0
for earlier, later in zip(reversed(progesterone_values[:-1]), reversed(progesterone_values[1:])):
    if later < earlier:
        withdrawal_transitions += 1
    else:
        break
cross_cycle_jump = abs(
    float(kinetic_following_df["progesterone_ng_ml"].iloc[0])
    - float(kinetic_cycle_df["progesterone_ng_ml"].iloc[-1])
)

healthy_kinetics_df = pd.DataFrame(
    [
        {"metric": "Preovulatory estradiol width at >=80% maximum (days)", "value": peak_width_days},
        {"metric": "Luteal E2 peak / follicular E2 peak", "value": luteal_e2_peak_ratio},
        {"metric": "Progesterone width at >=75% maximum (days)", "value": progesterone_plateau_days},
        {"metric": "Progesterone peak offset from ovulation (days)", "value": progesterone_peak_day - ovulation_day},
        {"metric": "Progesterone 5 ng/mL rise offset (days)", "value": progesterone_rise_day - ovulation_day},
        {"metric": "Consecutive progesterone-decline transitions before bleeding", "value": withdrawal_transitions},
        {"metric": "Final-cycle progesterone / cycle maximum", "value": progesterone_values[-1] / max(progesterone_values)},
        {"metric": "Progesterone jump across cycle boundary (ng/mL)", "value": cross_cycle_jump},
    ]
)
healthy_summary_df = pd.DataFrame(
    [
        {"metric": "Diary days", "value": len(healthy_df)},
        {"metric": "Cycles captured", "value": len(healthy_cycles_df)},
        {"metric": "Mean cycle length (days)", "value": healthy_cycles_df["cycle_length"].mean()},
        {"metric": "Ovulatory cycle rate", "value": healthy_cycles_df["ovulatory"].mean()},
        {"metric": "Total ovulation events", "value": int(healthy_df["ovulation"].sum())},
        {"metric": "Total bleeding days", "value": int(healthy_df["uterine_bleeding"].sum())},
    ]
)

print(f"Wrote healthy-patient diary CSV to {ONE_HEALTHY_PATH}")


Wrote healthy-patient diary CSV to /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/oneHealthy.csv


## One healthy 30-year-old female

            | metric | value |
| --- | --- |
| Diary days | 1095 |
| Cycles captured | 39 |
| Mean cycle length (days) | 28.462 |
| Ovulatory cycle rate | 0.974 |
| Total ovulation events | 38 |
| Total bleeding days | 163 |


In [6]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(healthy_df["day_index"], healthy_df["estradiol_pg_ml"], color="#C44E52", linewidth=1.2)
axes[0].set_ylabel("Estradiol\n(pg/mL)")
axes[0].set_title("One healthy 30-year-old female over 3 years")

axes[1].plot(healthy_df["day_index"], healthy_df["progesterone_ng_ml"], color="#4C72B0", linewidth=1.2)
axes[1].set_ylabel("Progesterone\n(ng/mL)")

axes[2].bar(
    healthy_df.loc[healthy_df["uterine_bleeding"] == 1, "day_index"],
    1,
    width=1.0,
    color="#DD8452",
    label="Menses",
)
ovulation_days = healthy_df.loc[healthy_df["ovulation"] == 1, "day_index"]
axes[2].scatter(ovulation_days, [1.05] * len(ovulation_days), color="#55A868", marker="^", s=28, label="Ovulation")
axes[2].set_ylabel("Events")
axes[2].set_xlabel("Day index")
axes[2].set_ylim(0, 1.25)
axes[2].legend(frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig(HEALTHY_FIG, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Wrote healthy-patient figure to {HEALTHY_FIG}")

fig, axes = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True, gridspec_kw={"height_ratios": [3, 3, 1]})
x_values = list(range(1, len(kinetic_plot_df) + 1))
boundary_x = len(kinetic_cycle_df) + 0.5

axes[0].plot(x_values, kinetic_plot_df["estradiol_pg_ml"], color="#C44E52", linewidth=2.2)
axes[0].set_ylabel("Estradiol (pg/mL)")
axes[0].set_title("Complete ovulatory cycle and first day of the next cycle")

axes[1].plot(x_values, kinetic_plot_df["progesterone_ng_ml"], color="#4C72B0", linewidth=2.2)
axes[1].set_ylabel("Progesterone (ng/mL)")

bleeding_positions = [
    position
    for position, bleeding in zip(x_values, kinetic_plot_df["uterine_bleeding"])
    if int(bleeding) == 1
]
axes[2].bar(bleeding_positions, [1.0] * len(bleeding_positions), color="#DD8452", width=0.85, label="Bleeding")
ovulation_positions = [
    position
    for position, ovulation in zip(x_values, kinetic_plot_df["ovulation"])
    if int(ovulation) == 1
]
axes[2].scatter(ovulation_positions, [1.08] * len(ovulation_positions), marker="^", s=65, color="#55A868", label="Ovulation")
axes[2].set_ylabel("Events")
axes[2].set_xlabel("Day within displayed interval")
axes[2].set_ylim(0, 1.25)
axes[2].legend(frameon=False, loc="upper right")

for axis in axes:
    axis.axvline(boundary_x, color="#666666", linestyle="--", linewidth=1.2, label="Cycle boundary")
fig.suptitle("Hormone-waveform check: luteal E2 rebound and broad P4 summit", fontsize=15)
fig.tight_layout()
fig.savefig(HEALTHY_KINETICS_FIG, dpi=180, bbox_inches="tight")
plt.close(fig)

print(f"Wrote hormone-kinetic figure to {HEALTHY_KINETICS_FIG}")


Wrote healthy-patient figure to /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/examples/reports/running_example_one_healthy.png
Wrote hormone-kinetic figure to /Users/dgoldenh/Documents/GitHub/catamenial-epilepsy-sim/examples/reports/running_example_hormone_kinetics.png


## Healthy-patient plot

            ![Healthy patient figure](examples/reports/running_example_one_healthy.png)

            ### Hormone-kinetic verification

            The zoomed cycle below uses separate physical scales for estradiol and progesterone,
            shows the secondary luteal estradiol rise and broadened progesterone summit, and
            includes the first day of the following cycle so the absence of a vertical reset is
            directly visible. The waveform uses all daily LH-aligned Stricker serum medians; its
            subphase amplitudes are checked independently against Anckaert et al.'s 85-woman cohort.

            | metric | value |
| --- | --- |
| Preovulatory estradiol width at >=80% maximum (days) | 3 |
| Luteal E2 peak / follicular E2 peak | 0.617 |
| Progesterone width at >=75% maximum (days) | 6 |
| Progesterone peak offset from ovulation (days) | 7 |
| Progesterone 5 ng/mL rise offset (days) | 2 |
| Consecutive progesterone-decline transitions before bleeding | 7 |
| Final-cycle progesterone / cycle maximum | 0.034 |
| Progesterone jump across cycle boundary (ng/mL) | 0.02 |

            ![Hormone kinetic verification](examples/reports/running_example_hormone_kinetics.png)


## Full citations

The notebook uses the simulator's built-in calibration, held-out, context, and direction/range references plus four additional input-distribution references. Evidence roles are declared in the machine-readable validation report.

- Daniels K, Abma JC. Current contraceptive status among females ages 15-49: United States, 2022-2023. NCHS Data Brief. 2025 Aug;(539):1-11. doi:10.15620/cdc/174618. [https://www.cdc.gov/nchs/products/databriefs/db539.htm](https://www.cdc.gov/nchs/products/databriefs/db539.htm)
- Azziz R, Woods KS, Reyna R, Key TJ, Knochenhauer ES, Yildiz BO. The prevalence and features of the polycystic ovary syndrome in an unselected population. Journal of Clinical Endocrinology & Metabolism. 2004;89(6):2745-2749. doi:10.1210/jc.2003-032046. [https://pubmed.ncbi.nlm.nih.gov/15181052/](https://pubmed.ncbi.nlm.nih.gov/15181052/)
- Ju H, Jones M, Mishra G. The prevalence and risk factors of dysmenorrhea. Epidemiologic Reviews. 2014;36:104-113. doi:10.1093/epirev/mxt009. [https://pubmed.ncbi.nlm.nih.gov/24284871/](https://pubmed.ncbi.nlm.nih.gov/24284871/)
- Marcus JL, Snowden JM, Murray Horwitz ME, et al. Use of intrauterine devices and risk of human immunodeficiency virus acquisition among insured women in the United States. Clinical Infectious Diseases. 2020;70(10):2221-2223. doi:10.1093/cid/ciz791. [https://pubmed.ncbi.nlm.nih.gov/31412356/](https://pubmed.ncbi.nlm.nih.gov/31412356/)

### Simulator scientific-source registry

- **Primary healthy-cycle calibration target.** Li H, Gibson EA, Jukic AMZ, et al. Menstrual cycle length variation by demographic characteristics from the Apple Women's Health Study. NPJ Digital Medicine. 2023;6(1):100. doi:10.1038/s41746-023-00848-1. [https://pubmed.ncbi.nlm.nih.gov/37248288/](https://pubmed.ncbi.nlm.nih.gov/37248288/)
- **Phase-timing and bleeding calibration target.** Bull JR, Rowland SP, Scherwitzl EB, et al. Real-world menstrual cycle characteristics of more than 600,000 menstrual cycles. NPJ Digital Medicine. 2019;2:83. doi:10.1038/s41746-019-0152-7. [https://pubmed.ncbi.nlm.nih.gov/31482137/](https://pubmed.ncbi.nlm.nih.gov/31482137/)
- **Held-out external healthy-cycle cross-check.** Cunningham AC, Pal L, Wickham AP, et al. Chronicling menstrual cycle patterns across the reproductive lifespan with real-world data. Scientific Reports. 2024;14(1):10172. doi:10.1038/s41598-024-60373-3. [https://pubmed.ncbi.nlm.nih.gov/38702411/](https://pubmed.ncbi.nlm.nih.gov/38702411/)
- **Daily hormone-shape calibration and face-validity target.** Stricker R, Eberhart R, Chevailler MC, et al. Establishment of detailed reference values for luteinizing hormone, follicle stimulating hormone, estradiol, and progesterone during different phases of the menstrual cycle on the Abbott ARCHITECT analyzer. Clinical Chemistry and Laboratory Medicine. 2006;44(7):883-887. doi:10.1515/CCLM.2006.160. [https://pubmed.ncbi.nlm.nih.gov/16776638/](https://pubmed.ncbi.nlm.nih.gov/16776638/)
- **Long-follicular-phase estradiol morphology source.** Harlow SD, Baird DD, Weinberg CR, Wilcox AJ. Urinary oestrogen patterns in long follicular phases. Human Reproduction. 2000;15(1):11-16. doi:10.1093/humrep/15.1.11. [https://pubmed.ncbi.nlm.nih.gov/10611180/](https://pubmed.ncbi.nlm.nih.gov/10611180/)
- **Independent long-cycle hormone-timing context.** Mumford SL, Steiner AZ, Pollack AZ, et al. The utility of menstrual cycle length as an indicator of cumulative hormonal exposure. Journal of Clinical Endocrinology & Metabolism. 2012;97(10):E1871-E1879. doi:10.1210/jc.2012-1350. [https://pubmed.ncbi.nlm.nih.gov/22837188/](https://pubmed.ncbi.nlm.nih.gov/22837188/)
- **Independent serum-hormone subphase validation target.** Anckaert E, Jank A, Petzold J, et al. Extensive monitoring of the natural menstrual cycle using the serum biomarkers estradiol, luteinizing hormone and progesterone. Practical Laboratory Medicine. 2021;25:e00211. doi:10.1016/j.plabm.2021.e00211. [https://pubmed.ncbi.nlm.nih.gov/33869706/](https://pubmed.ncbi.nlm.nih.gov/33869706/)
- **Normal/abnormal uterine-bleeding terminology and range context.** Fraser IS, Critchley HOD, Broder M, Munro MG. The FIGO recommendations on terminologies and definitions for normal and abnormal uterine bleeding. Seminars in Reproductive Medicine. 2011;29(5):383-390. doi:10.1055/s-0031-1287662. [https://pubmed.ncbi.nlm.nih.gov/22065325/](https://pubmed.ncbi.nlm.nih.gov/22065325/)
- **Direction-only PCOS cycle-length and variability check.** Mortimer R, Asokan G, Baird DD, et al. Variability of menstrual cycles by age, polycystic ovary syndrome, and early-life cycle irregularity in the Apple Women's Health Study. American Journal of Obstetrics and Gynecology. 2026;234(4):1042-1069. doi:10.1016/j.ajog.2025.11.031. [https://pubmed.ncbi.nlm.nih.gov/41297783/](https://pubmed.ncbi.nlm.nih.gov/41297783/)
- **Direction-only PCOS ovulatory-status and steroid-pattern check.** Doi SAR, Al-Zaid M, Towers PA, Scott CJ, Al-Shoumer KAS. Irregular cycles and steroid hormones in polycystic ovary syndrome. Human Reproduction. 2005;20(9):2402-2408. doi:10.1093/humrep/dei093. [https://pubmed.ncbi.nlm.nih.gov/15932911/](https://pubmed.ncbi.nlm.nih.gov/15932911/)
- **Direction-only PCOS follicular and hormone-pattern context.** Jarrett BY, Vanden Brink H, Oldfield AL, Lujan ME. Ultrasound characterization of disordered antral follicle development in women with polycystic ovary syndrome. Journal of Clinical Endocrinology & Metabolism. 2020;105(11):e3847-e3861. doi:10.1210/clinem/dgaa515. [https://pubmed.ncbi.nlm.nih.gov/32785651/](https://pubmed.ncbi.nlm.nih.gov/32785651/)
- **Direction-only peri-menarche cycle-length and regularity check.** World Health Organization Task Force on Adolescent Reproductive Health. World Health Organization multicenter study on menstrual and ovulatory patterns in adolescent girls. II. Longitudinal study of menstrual patterns in the early postmenarcheal period, duration of bleeding episodes and menstrual cycles. Journal of Adolescent Health Care. 1986;7(4):236-244. [https://pubmed.ncbi.nlm.nih.gov/3721946/](https://pubmed.ncbi.nlm.nih.gov/3721946/)
- **Direction-only peri-menarche ovulation and hormone-pattern check.** Venturoli S, Porcu E, Fabbri R, et al. Menstrual irregularities in adolescents: hormonal pattern and ovarian morphology. Hormone Research. 1986;24(4):269-279. doi:10.1159/000180567. [https://pubmed.ncbi.nlm.nih.gov/3491030/](https://pubmed.ncbi.nlm.nih.gov/3491030/)
- **Peri-menarche hormone-pattern context and limitation.** Zhang K, Pollack S, Ghods A, et al. Onset of ovulation after menarche in girls: a longitudinal study. Journal of Clinical Endocrinology & Metabolism. 2008;93(4):1186-1194. doi:10.1210/jc.2007-1846. [https://pubmed.ncbi.nlm.nih.gov/18252789/](https://pubmed.ncbi.nlm.nih.gov/18252789/)
- **Direction-only perimenopause cycle and hormone-pattern check.** Santoro N, Randolph JF Jr. Reproductive hormones and the menopause transition. Obstetrics and Gynecology Clinics of North America. 2011;38(3):455-466. doi:10.1016/j.ogc.2011.05.004. [https://pubmed.ncbi.nlm.nih.gov/21961713/](https://pubmed.ncbi.nlm.nih.gov/21961713/)
- **Direction/range check for combined oral-contraceptive regimens.** Edelman AB, Gallo MF, Jensen JT, et al. Continuous or extended cycle vs. cyclic use of combined hormonal contraceptives for contraception. Cochrane Database of Systematic Reviews. 2014;(7):CD004695. doi:10.1002/14651858.CD004695.pub3. [https://pubmed.ncbi.nlm.nih.gov/25072731/](https://pubmed.ncbi.nlm.nih.gov/25072731/)
- **Direction/range check for long-term levonorgestrel-IUD ovarian function.** Xiao B, Zeng T, Wu S, Sun H, Xiao N. Effect of levonorgestrel-releasing intrauterine device on hormonal profile and menstrual pattern after long-term use. Contraception. 1995;51(6):359-365. doi:10.1016/0010-7824(95)00102-G. [https://pubmed.ncbi.nlm.nih.gov/7554977/](https://pubmed.ncbi.nlm.nih.gov/7554977/)
- **Direction-only copper-IUD ovulation and cycle-length check.** Faundes A, Segal SJ, Adejuwon CA, et al. The menstrual cycle in women using an intrauterine device. Fertility and Sterility. 1980;34(5):427-430. doi:10.1016/S0015-0282(16)45131-9. [https://pubmed.ncbi.nlm.nih.gov/7439408/](https://pubmed.ncbi.nlm.nih.gov/7439408/)
- **Direction-only copper-IUD bleeding check.** Malmqvist R, Petersohn L, Bengtsson LP. Menstrual bleeding with copper-covered intrauterine contraceptive devices. Contraception. 1974;9(6):627-633. doi:10.1016/0010-7824(74)90048-1. [https://pubmed.ncbi.nlm.nih.gov/4448089/](https://pubmed.ncbi.nlm.nih.gov/4448089/)
- **Direction-only primary-dysmenorrhea phenotype check.** Dawood MY. Primary dysmenorrhea: advances in pathogenesis and management. Obstetrics and Gynecology. 2006;108(2):428-441. doi:10.1097/01.AOG.0000230214.26638.0c. [https://pubmed.ncbi.nlm.nih.gov/16880317/](https://pubmed.ncbi.nlm.nih.gov/16880317/)
